In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from PIL import Image, ImageOps


# --- 1. DATA PREPROCESSING FUNCTIONS ---

def read_and_import_dataset():
    """
    Reads the Kaggle 'train.csv' file and splits it into
    a new training set and a validation set.
    """
    print("Reading dataset...")
    try:
        training_data = pd.read_csv("train.csv")
    except FileNotFoundError:
        print("Error: 'train.csv' not found. Make sure it's in the same directory.")
        return None, None, None, None

    # First, split the full training data into features (X) and labels (y)
    X_full = training_data.drop('label', axis=1)
    y_full = training_data['label']

    # Now, split this full set into a new (smaller) training set
    # and a validation set (which we will use to test our model)
    # Let's use 10% of the data for validation (4,200 samples)
    X_train, X_val, y_train, y_val = train_test_split(
        X_full, y_full, test_size=0.1, random_state=42
    )

    print("Dataset split into training and validation sets.")
    return X_train, y_train, X_val, y_val


def normalize_and_convert(X_train, X_val):
    """
    Normalizes pixel values from 0-255 to 0-1 and converts
    Pandas DataFrames to NumPy arrays.
    """
    # Normalizing - Converting the int values from 0-255 to float 0-1.
    X_train_norm = X_train / 255.0
    X_val_norm = X_val / 255.0

    # Convert from pandas DataFrame to NumPy array for math operations
    X_train_numpy = X_train_norm.to_numpy()
    X_val_numpy = X_val_norm.to_numpy()

    return X_train_numpy, X_val_numpy


def one_hot_encode(y):
    """
    Converts a 1D array of labels (e.g., [5, 0, 4]) into a 2D
    one-hot encoded array.
    """
    # First, convert the pandas Series to a NumPy array
    y_numpy = y.to_numpy()
    num_samples = y_numpy.shape[0]

    # Create an empty array of zeros: (num_samples, 10)
    y_one_hot = np.zeros((num_samples, 10))

    # Use advanced indexing to set the correct class to 1
    y_one_hot[np.arange(num_samples), y_numpy] = 1

    return y_one_hot


# --- 2. NEURAL NETWORK BUILDING BLOCKS ---

def init_params():
    """
    Initializes the weights (W) and biases (b) for a
    784 -> 64 -> 10 network.
    """
    input_nodes = 784
    hidden_nodes = 64
    output_nodes = 10

    # Initialize weights with small random numbers
    W1 = np.random.randn(input_nodes, hidden_nodes) * 0.01
    b1 = np.zeros((1, hidden_nodes))

    # Do the same for the second layer
    W2 = np.random.randn(hidden_nodes, output_nodes) * 0.01
    b2 = np.zeros((1, output_nodes))

    print("Parameters initialized.")
    return W1, b1, W2, b2


def sigmoid_function(Z):
    """The sigmoid activation function."""
    return 1 / (1 + np.exp(-Z))


def softmax(Z):
    """The softmax activation function (numerically stable)."""
    # Shift Z by its max value for numerical stability
    exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)


def forward_prop(X, W1, b1, W2, b2):
    """
    Performs one forward pass, returning all intermediate values
    for backpropagation.
    """
    Z1 = np.dot(X, W1) + b1
    A1 = sigmoid_function(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)

    # Return all intermediate steps for backprop
    return Z1, A1, Z2, A2


def calculate_loss(A2, Y):
    """
    Calculates the cross-entropy loss.
    A2 is predictions, Y is one-hot true labels.
    """
    m = Y.shape[0]  # Number of samples

    # Clip values to avoid log(0)
    A2_clipped = np.clip(A2, 1e-9, 1 - 1e-9)

    # Calculate the loss
    log_likelihoods = Y * np.log(A2_clipped)
    total_loss = np.sum(log_likelihoods)
    loss = - (1 / m) * total_loss

    return loss


def backward_prop(X, Y, Z1, A1, Z2, A2, W2):
    """
    Performs one backward pass, calculating the gradients
    for all parameters.
    """
    m = Y.shape[0]  # Number of samples

    # 1. Start at the End (Layer 2)
    dZ2 = A2 - Y  # (num_samples, 10)

    # 2. Gradients for W2 and b2
    dW2 = (1 / m) * np.dot(A1.T, dZ2)  # (64, 10)
    db2 = (1 / m) * np.sum(dZ2, axis=0, keepdims=True)  # (1, 10)

    # 3. Pass the Error to Layer 1
    dA1 = np.dot(dZ2, W2.T)  # (num_samples, 64)
    dZ1 = dA1 * (A1 * (1 - A1))  # (num_samples, 64)

    # 4. Gradients for W1 and b1
    dW1 = (1 / m) * np.dot(X.T, dZ1)  # (784, 64)
    db1 = (1 / m) * np.sum(dZ1, axis=0, keepdims=True)  # (1, 64)

    return dW1, db1, dW2, db2


def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate):
    """
    Updates parameters using the gradient descent update rule.
    """
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2

    return W1, b1, W2, b2


# --- 3. ACCURACY & TRAINING FUNCTIONS ---

def get_predictions(A2):
    """
    Converts softmax probabilities (A2) into final
    digit predictions (0-9).
    """
    # Find the index with the highest probability
    return np.argmax(A2, axis=1)


def get_accuracy(predictions, Y_labels):
    """
    Calculates the accuracy by comparing predictions
    to the true (non-one-hot) labels.
    """
    # Y_labels should be the 1D array of digits [5, 0, 4, ...]
    return np.sum(predictions == Y_labels) / Y_labels.shape[0]


def train_model(X_train, Y_train_one_hot, X_val, Y_val_series, W1, b1, W2, b2, learning_rate, epochs):
    """
    The main training loop that runs gradient descent.
    """

    # We need the non-one-hot validation labels to check accuracy
    Y_val_numpy = Y_val_series.to_numpy()

    print("\n--- Starting Training ---")
    print(f"Learning Rate: {learning_rate}")
    print(f"Epochs: {epochs}")

    for i in range(epochs):
        # --- 1. Forward Propagation (on training data) ---
        Z1, A1, Z2, A2 = forward_prop(X_train, W1, b1, W2, b2)

        # --- 2. Calculate Loss (on training data) ---
        loss = calculate_loss(A2, Y_train_one_hot)

        # --- 3. Backward Propagation ---
        dW1, db1, dW2, db2 = backward_prop(X_train, Y_train_one_hot, Z1, A1, Z2, A2, W2)

        # --- 4. Update Parameters ---
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate)

        # --- 5. Print Progress (e.g., every 100 epochs) ---
        if i % 100 == 0:
            print(f"\nEpoch: {i}")
            print(f"Loss: {loss:.4f}")  # Print loss with 4 decimal places

            # --- Check accuracy on the VALIDATION set ---
            _, _, _, A2_val = forward_prop(X_val, W1, b1, W2, b2)
            val_predictions = get_predictions(A2_val)
            accuracy = get_accuracy(val_predictions, Y_val_numpy)
            print(f"Validation Accuracy: {accuracy * 100:.2f}%")

    return W1, b1, W2, b2


# --- 4. NEW FUNCTIONS FOR CUSTOM IMAGE PREDICTION ---

def preprocess_image(image_path):
    """
    Loads a user-created image, inverts it, resizes it to 28x28,
    and formats it to match the (1, 784) MNIST data.
    """
    try:
        # 1. Load the image
        img = Image.open(image_path)
    except FileNotFoundError:
        print(f"Error: Image '{image_path}' not found.")
        return None

    # 2. Convert to grayscale ('L' mode)
    img = img.convert('L')

    # 3. Invert the colors
    # MNIST = white digit, black background. Drawing = black digit, white background.
    img = ImageOps.invert(img)

    # 4. Resize to 28x28
    img = img.resize((28, 28), Image.Resampling.LANCZOS)

    # 5. Convert to NumPy array
    img_data = np.asarray(img)

    # 6. Normalize
    img_norm = img_data / 255.0

    # 7. Flatten to (1, 784) - a single sample with 784 features
    img_flat = img_norm.reshape(1, 784)

    return img_flat


def make_prediction(X_new, W1, b1, W2, b2):
    """
    Runs a single forward pass on new data (X_new) using
    the trained parameters and returns the predicted digit.
    """
    # Run the forward pass
    _, _, _, A2 = forward_prop(X_new, W1, b1, W2, b2)

    # Get the prediction (the index with the highest probability)
    predictions = get_predictions(A2)

    # Return the single digit
    return predictions[0]


# --- 5. MAIN EXECUTION ---

# Use if __name__ == "__main__": to make sure this code
# only runs when the script is executed directly.
if __name__ == "__main__":

    # --- 1. Load and Preprocess Data ---
    X_train_df, y_train_series, X_val_df, y_val_series = read_and_import_dataset()

    if X_train_df is not None:
        print(f"Original training images shape: {X_train_df.shape}")
        print(f"Original validation images shape: {X_val_df.shape}")

        # Normalize and convert image data to NumPy arrays
        X_train_np, X_val_np = normalize_and_convert(X_train_df, X_val_df)
        print(f"NumPy training images shape: {X_train_np.shape}")
        print(f"NumPy validation images shape: {X_val_np.shape}")

        # One-hot encode the labels (Pandas -> NumPy)
        y_train_one_hot = one_hot_encode(y_train_series)
        y_val_one_hot = one_hot_encode(y_val_series)
        print(f"NumPy training labels shape: {y_train_one_hot.shape}")
        print(f"NumPy validation labels shape: {y_val_one_hot.shape}")

        # --- 2. Initialize Network ---
        W1, b1, W2, b2 = init_params()
        print(f"W1 shape: {W1.shape}, b1 shape: {b1.shape}")
        print(f"W2 shape: {W2.shape}, b2 shape: {b2.shape}")

        # --- 3. Set Hyperparameters ---
        learning_rate = 0.1
        epochs = 4000  # Start with 2000, you can increase this

        # --- 4. Run the Training ---
        trained_W1, trained_b1, trained_W2, trained_b2 = train_model(
            X_train_np,  # Training images
            y_train_one_hot,  # Training labels (one-hot)
            X_val_np,  # Validation images
            y_val_series,  # Validation labels (original Series)
            W1, b1, W2, b2,  # Initial parameters
            learning_rate,  # Hyperparameter
            epochs  # Hyperparameter
        )

        print("\n--- Training Complete ---")

Reading dataset...
Dataset split into training and validation sets.
Original training images shape: (37800, 784)
Original validation images shape: (4200, 784)
NumPy training images shape: (37800, 784)
NumPy validation images shape: (4200, 784)
NumPy training labels shape: (37800, 10)
NumPy validation labels shape: (4200, 10)
Parameters initialized.
W1 shape: (784, 64), b1 shape: (1, 64)
W2 shape: (64, 10), b2 shape: (1, 10)

--- Starting Training ---
Learning Rate: 0.1
Epochs: 4000

Epoch: 0
Loss: 2.3042
Validation Accuracy: 9.71%

Epoch: 100
Loss: 2.2821
Validation Accuracy: 12.95%

Epoch: 200
Loss: 2.1316
Validation Accuracy: 39.12%

Epoch: 300
Loss: 1.6679
Validation Accuracy: 56.67%

Epoch: 400
Loss: 1.2292
Validation Accuracy: 69.07%

Epoch: 500
Loss: 0.9657
Validation Accuracy: 76.62%

Epoch: 600
Loss: 0.7993
Validation Accuracy: 81.24%

Epoch: 700
Loss: 0.6884
Validation Accuracy: 83.67%

Epoch: 800
Loss: 0.6117
Validation Accuracy: 85.05%

Epoch: 900
Loss: 0.5560
Validation Acc

In [8]:
        # --- 5. TEST YOUR OWN IMAGE ---
        print("\n--- Testing a Custom Image ---")

        # --- IMPORTANT ---
        # 1. Create a 28x28 (or larger) image in MS Paint.
        # 2. Draw a SINGLE digit (like '7') in BLACK on a WHITE background.
        # 3. Try to center it.
        # 4. Save it as "my_digit.png" in the SAME folder as this script.
        # ---------------

        image_path = "my_digit.png"
        my_image = preprocess_image(image_path)

        if my_image is not None:
            # Make a prediction using the *trained* parameters
            my_prediction = make_prediction(my_image, trained_W1, trained_b1, trained_W2, trained_b2)

            print(f"The model predicts your image '{image_path}' is a: {my_prediction}")
        else:
            print("Skipping custom image prediction due to file error.")


--- Testing a Custom Image ---
The model predicts your image 'my_digit.png' is a: 3
